In [ ]:
# [Cell 0]
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/football-tracker")
else:
    DRIVE = Path("drive_checkpoints")
    # Local run: keep model, torch, and HF downloads inside the project.
    PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "__init__.py").exists())
    sys.path.insert(0, str(PROJECT_DIR))
    import src  # noqa: F401
DRIVE.mkdir(parents=True, exist_ok=True)

MODELS_DIR = DRIVE / "models"
MODELS_DIR.mkdir(exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
print(f"Checkpoints: {DRIVE}")
print(f"Models: {MODELS_DIR}")

In [ ]:
# [Cell 1]
!pip install -q ultralytics huggingface_hub
from ultralytics import YOLO, settings
settings.update({"runs_dir": str(DRIVE / "runs")})
print("Ultralytics ready. Runs -> Drive.")

In [ ]:
# [Cell 2]
from huggingface_hub import snapshot_download

dataset_path = DATA_DIR / "football_player"
snapshot_download(
    "martinjolif/football-player-detection",
    repo_type="dataset",
    local_dir=str(dataset_path),
)
print(f"Downloaded to {dataset_path}")
print("Files:", list(dataset_path.iterdir()))

In [ ]:
# [Cell 3]
# Source dataset has 4 classes [ball, goalkeeper, player, referee]. Production
# uses the detector with classes=[0], so build a single-class "player" dataset
# (goalkeeper + player -> 0; ball/referee boxes dropped).
import shutil
import yaml

src_yaml = yaml.safe_load(open(dataset_path / "data.yaml")) if (dataset_path / "data.yaml").exists() else {}
names = src_yaml.get("names", ["ball", "goalkeeper", "player", "referee"])
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
keep_ids = {i for i, n in enumerate(names) if n in ("goalkeeper", "player")}

player_dir = DATA_DIR / "football_player_1cls"
for split in ("train", "valid"):
    (player_dir / split / "images").mkdir(parents=True, exist_ok=True)
    (player_dir / split / "labels").mkdir(parents=True, exist_ok=True)
    for img in (dataset_path / split / "images").iterdir():
        shutil.copy(img, player_dir / split / "images" / img.name)
        lbl = dataset_path / split / "labels" / f"{img.stem}.txt"
        rows = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if parts and int(parts[0]) in keep_ids:
                    rows.append(" ".join(["0", *parts[1:]]))
        (player_dir / split / "labels" / f"{img.stem}.txt").write_text("\n".join(rows))

data_yaml = player_dir / "data.yaml"
yaml.dump({
    "path": str(player_dir.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "nc": 1,
    "names": ["player"],
}, open(data_yaml, "w"))
print(f"Created {data_yaml} (source classes kept: {sorted(keep_ids)} of {names})")

In [ ]:
# [Cell 4]
model = YOLO("yolo26s.pt")
results = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=960,  # matches production inference size
    batch=16,
    name="football_player",
    pretrained=True,
    device=0,
    save_period=5,
    hsv_h=0.015,
    hsv_s=0.3,
    hsv_v=0.2,
    degrees=5,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)
print(f"Best model: {results.save_dir}")

In [ ]:
# [Cell 5]
from pathlib import Path

runs = sorted((DRIVE / "runs" / "detect").glob("football_player*"), reverse=True)
best_pt = runs[0] / "weights" / "best.pt"
model = YOLO(str(best_pt))
model.export(format="onnx", imgsz=960)  # production imgsz
print(f"Exported ONNX from {best_pt}")

In [ ]:
# [Cell 6]
import shutil

runs = sorted((DRIVE / "runs" / "detect").glob("football_player*"), reverse=True)
src_pt = runs[0] / "weights" / "best.pt"
src_onnx = Path(str(src_pt).replace(".pt", ".onnx"))

shutil.copy(src_pt, MODELS_DIR / "player_detector.pt")
if src_onnx.exists():
    shutil.copy(src_onnx, MODELS_DIR / "player_detector.onnx")

for f in MODELS_DIR.iterdir():
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")
print("Saved to Drive.")